In [ ]:
from google.colab import files
files.upload()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ReadCSVJSON") \
    .getOrCreate()

In [ ]:
csv_path = "/content/orders_large_bad.csv"

df_csv = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

df_csv.show(5)

In [ ]:
json_path = "/content/orders_large_bad.json.json"

df_json = spark.read.json(json_path)

df_json.show(5)

In [ ]:
!ls /content

PHASE 1 — INGESTION & FIRST INSPECTION
1) Read the CSV file into a DataFrame

In [ ]:
df_csv_inferred = spark.read.option("header", True).csv(csv_path)

2) Disable schema inference and read everything as string

In [ ]:

df_csv_raw = spark.read.option("header", True)\
    .option("inferSchema", False)\
    .csv(csv_path)


df_csv = df_csv_raw.select([F.col(c).cast(StringType()).alias(c) for c in df_csv_raw.columns])


3) Print schema and record count

In [ ]:

df_csv.printSchema()
print("CSV row count:", df_csv.count())


4) Display 20 random rows

In [ ]:
df_csv.orderBy(F.rand()).show(20, truncate=False)

5) Identify at least 5 data quality issues by observation

In [ ]:

quality_issues = {
    "amount_invalid_values": df_csv.filter(F.col("amount").rlike("(?i)invalid")).count(),
    "amount_commas": df_csv.filter(F.col("amount").rlike(".*,.*")).count(),
    "amount_blank": df_csv.filter((F.col("amount").isNull()) | (F.col("amount") == "")).count(),
    "order_date_invalid_token": df_csv.filter(F.col("order_date").rlike("(?i)invalid")).count(),
    "mixed_date_formats": df_csv.select("order_date").distinct().filter(
        F.col("order_date").rlike(r".*/.*/.*|.*-.*-.*") # simple heuristic
    ).count(),
    "case_inconsistent_city": df_csv.select("city").distinct().count(), # will compare later
    "status_values": df_csv.select("status").distinct().collect(),
    "product_blank": df_csv.filter((F.col("product").isNull()) | (F.col("product") == "")).count()
}
quality_issues


6) Read the JSON file and compare schema and row count with CSV

In [ ]:

# Read JSON as raw strings (no schema inference; JSON reader produces strings by default)
df_json = spark.read.option("multiLine", True).json(json_path)

df_json.printSchema()
print("JSON row count:", df_json.count())

print("CSV vs JSON row count:", df_csv.count(), df_json.count())
print("CSV columns:", df_csv.columns)
print("JSON columns:", df_json.columns)



PHASE 2 — SCHEMA ENFORCEMENT & VALIDATION
7) Define an explicit schema using StructType

In [ ]:

explicit_schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city",        StringType(), True),
    StructField("category",    StringType(), True),
    StructField("product",     StringType(), True),
    StructField("amount",      StringType(), True),   # keep as string before cleaning
    StructField("order_date",  StringType(), True),   # keep as string before parsing
    StructField("status",      StringType(), True),
])



8)Re-read the CSV using the defined schema

In [ ]:

df_csv_typed = spark.read.option("header", True)\
    .schema(explicit_schema)\
    .csv(csv_path)
df_csv_typed.printSchema()


9) Identify rows that fail schema expectations

In [ ]:

expected_status = ["Completed", "Cancelled"]
invalid_status_rows = df_csv_typed.filter(~F.col("status").isin(expected_status))
invalid_status_rows.count(), invalid_status_rows.limit(5).show(truncate=False)


10) Explain why schema inference is dangerous at scale


Inference can misinterpret columns (e.g., amount with commas → string vs int) and flip types between batches.
On large data, inference scans the data, adding latency and potential type drift (e.g., order_date mixed formats).
Explicit schemas ensure deterministic pipelines, stable metadata, and fewer runtime surprises during merges/streaming.

PHASE 3 — STRING CLEANING & STANDARDIZATION
11) Trim leading and trailing spaces from all string columns

In [ ]:

def trim_all(df):
    return df.select([F.trim(F.col(c)).alias(c) for c in df.columns])

df_trim = trim_all(df_csv_typed)


12) Standardize city, category, and product values

In [ ]:
city_map = {
    "bangalore":"Bangalore","bengaluru":"Bangalore",
    "mumbai":"Mumbai","bombay":"Mumbai",
    "delhi":"Delhi","new delhi":"Delhi",
    "kolkata":"Kolkata","calcutta":"Kolkata",
    "pune":"Pune","poona":"Pune",
    "hyderabad":"Hyderabad","secunderabad":"Hyderabad",
    "chennai":"Chennai","madras":"Chennai"
}

cat_map = {"grocery":"Grocery","electronics":"Electronics","home":"Home","fashion":"Fashion"}

# The UDF will now directly access the maps from its closure
def normalize_func(s, mapping_dict):
    return mapping_dict.get((s or "").strip().lower(), (s or "").strip().title())

normalize_city = F.udf(lambda s: normalize_func(s, city_map), StringType())
normalize_category = F.udf(lambda s: normalize_func(s, cat_map), StringType())

df_std = (df_trim
    .withColumn("city",     normalize_city(F.col("city")))
    .withColumn("category", normalize_category(F.col("category")))
    .withColumn("product",  F.initcap(F.col("product")))  # simple: "tshirt" -> "Tshirt"
)

13) Convert all categorical columns to a consistent case

In [ ]:

df_cat_case = (df_std
    .withColumn("city",     F.initcap(F.col("city")))
    .withColumn("category", F.initcap(F.col("category")))
    .withColumn("status",   F.initcap(F.col("status")))
)


14) Identify how many distinct city values existed before vs after cleaning

In [ ]:

before_city_distinct = df_csv_typed.select("city").distinct().count()
after_city_distinct  = df_cat_case.select("city").distinct().count()
print("Distinct cities — before:", before_city_distinct, "after:", after_city_distinct)


PHASE 4 — AMOUNT CLEANING (CRITICAL)
15) Identify invalid values in the amount column

In [ ]:

invalid_amount = df_cat_case.filter(F.col("amount").rlike("(?i)invalid|[^0-9,]"))


16) Remove commas from numeric strings

In [ ]:
df_amt = df_cat_case.withColumn("amount_nocomma", F.regexp_replace(F.col("amount"), ",", ""))

17) Convert amount to IntegerType safely

In [ ]:

df_amt2 = df_amt.withColumn(
    "amount_int",
    F.when(F.col("amount_nocomma").rlike("^[0-9]+$"), F.col("amount_nocomma").cast(IntegerType()))
     .otherwise(F.lit(None).cast(IntegerType()))
)


18) Handle empty, null, and invalid values explicitly

In [ ]:

df_amt3 = (df_amt2
    .withColumn("amount_clean",
        F.when((F.col("amount_int").isNull()) | (F.col("amount_int") <= 0), F.lit(None).cast(IntegerType()))
         .otherwise(F.col("amount_int"))
    )
)


19) Count how many records were affected during amount cleaning

In [ ]:

affected_amt = df_amt3.filter(F.col("amount_clean").isNull()).count()
print("Records with invalid/empty/<=0 amount:", affected_amt)



PHASE 5 — DATE PARSING & NORMALIZATION
20) Identify all date formats present in order_date

In [ ]:
df_date_formats = df_amt3.select(
    "order_date",
    F.when(F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), F.lit("YYYY-MM-DD"))
     .when(F.col("order_date").rlike(r"^\d{4}/\d{2}/\d{2}$"), F.lit("YYYY/MM/DD"))
     .when(F.col("order_date").rlike(r"^\d{2}/\d{2}/\d{4}$"), F.lit("DD/MM/YYYY"))
     .when(F.col("order_date").rlike(r"^\d{2}-\d{2}-\d{4}$"), F.lit("DD-MM-YYYY"))
     .when(F.col("order_date").rlike("(?i)invalid"), F.lit("INVALID"))
     .otherwise(F.lit("UNKNOWN")).alias("format_type")
).groupBy("format_type").agg(F.count("*").alias("count"))
df_date_formats.show(truncate=False)

21) Parse valid dates into DateType

In [ ]:
from datetime import datetime
from pyspark.sql.types import DateType

@F.udf(DateType())
def multi_parse_udf(date_str):
    if date_str is None:
        return None
    formats = [
        "%Y-%m-%d", # YYYY-MM-DD
        "%Y/%m/%d", # YYYY/MM/DD
        "%d/%m/%Y", # DD/MM/YYYY
        "%d-%m-%Y"  # DD-MM-YYYY
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt).date()
        except ValueError:
            continue
    return None

df_dates = df_amt3.withColumn("order_date_clean", multi_parse_udf(F.col("order_date")))

22) Handle invalid dates gracefully

In [ ]:

df_dates_flagged = df_dates.withColumn(
    "date_invalid_flag",
    F.when(F.col("order_date_clean").isNull(), F.lit(True)).otherwise(F.lit(False))
)


23) Create a clean order_date_clean column

Done above (order_date_clean)

In [ ]:
24) Count records with invalid dates

In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, coalesce, trim, lower

# Parse dates with ALL possible formats
df_dates_flagged = df_csv.withColumn(
    "order_date_parsed",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd"),
        to_date(col("order_date"), "dd/MM/yyyy")
    )
).withColumn(
    "date_invalid_flag",
    col("order_date_parsed").isNull()
)

# Count invalid dates (SAFE)
invalid_date_count = df_dates_flagged.filter(col("date_invalid_flag")).count()

print("Invalid/Unparsed dates:", invalid_date_count)

PHASE 6 — BUSINESS FILTERING & DEDUPLICATION
25) Identify duplicate order_id values

In [ ]:

dupes = (df_dates_flagged
    .groupBy("order_id")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)
dupes.count(), dupes.orderBy(F.col("cnt").desc()).show(20, truncate=False)


26) Remove duplicate orders safely

In [ ]:

# Keep the first non-null amount_clean, earliest parsed date, etc.
window = Window.partitionBy("order_id").orderBy(F.col("order_date_clean").asc_nulls_last())
df_dedup = (df_dates_flagged
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)


27) Keep only records with status = Completed

In [ ]:
df_completed = df_dedup.filter(F.col("status") == "Completed")

28) Validate record counts before and after filtering

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, coalesce, trim, lower


df_base = df_csv.withColumn(
    "order_date_clean",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd")
    )
).withColumn(
    "amount_clean",
    col("amount").cast("double")
).filter(
    col("order_date_clean").isNotNull()
)

df_cat_case = df_base.withColumn(
    "category_clean",
    lower(trim(col("category")))
)


df_dedup = df_cat_case.dropDuplicates(["order_id"])

df_completed = df_dedup.filter(col("status") == "Completed")


print(
    "Raw:", df_csv.count(),
    "After clean:", df_base.count(),
    "After cat std:", df_cat_case.count(),
    "Dedup:", df_dedup.count(),
    "Completed:", df_completed.count()
)


PHASE 7 — PERFORMANCE & PARTITION AWARENESS
29) Check the default number of partitions

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, coalesce

# Rebuild df_completed safely (FIX)
df_completed = df_csv.withColumn(
    "order_date_clean",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd")
    )
).filter(col("order_date_clean").isNotNull()) \
.withColumn("amount_clean", col("amount").cast("double"))



30) Run a heavy groupBy and observe execution time

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, coalesce

# Clean date (handles yyyy-MM-dd and yyyy/MM/dd)
df_clean = df_csv.withColumn(
    "order_date_clean",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd")
    )
)

# Remove rows with invalid dates
df_clean = df_clean.filter(col("order_date_clean").isNotNull())

# Clean amount column
df_clean = df_clean.withColumn(
    "amount_clean",
    col("amount").cast("double")
)

# Aggregate total revenue by city
city_rev = df_clean.groupBy("city") \
    .agg(F.sum("amount_clean").alias("total_revenue"))

city_rev.show()

31) Use explain(True) to identify shuffle stages

In [ ]:
city_rev.explain(True)

32) Repartition the DataFrame by city

In [ ]:

df_by_city = df_completed.repartition("city")
print("repartitioned partitions:", df_by_city.rdd.getNumPartitions())


33) Compare execution plans before and after repartition

In [ ]:

df_completed.groupBy("city").agg(F.sum("amount_clean")).explain(True)
df_by_city.groupBy("city").agg(F.sum("amount_clean")).explain(True)


PHASE 8 — ANALYTICS ON LARGE DATA
34) Calculate total revenue per city

In [ ]:

rev_city = df_completed.groupBy("city").agg(F.sum("amount_clean").alias("revenue")).orderBy(F.col("revenue").desc())
rev_city.show(50, truncate=False)


35) Calculate total revenue per category

In [ ]:

rev_cat = df_completed.groupBy("category").agg(F.sum("amount_clean").alias("revenue")).orderBy(F.col("revenue").desc())
rev_cat.show(50, truncate=False)


36) Calculate total revenue per product

In [ ]:

rev_prod = df_completed.groupBy("product").agg(F.sum("amount_clean").alias("revenue")).orderBy(F.col("revenue").desc())
rev_prod.show(50, truncate=False)


37) Identify top 10 products by revenue

In [ ]:

top10_products = rev_prod.limit(10)
top10_products.show(truncate=False)


38) Calculate average order value per city

In [ ]:

aov_city = df_completed.groupBy("city").agg(F.avg("amount_clean").alias("avg_order_value")).orderBy(F.col("avg_order_value").desc())
aov_city.show(50, truncate=False)


PHASE 9 — WINDOW FUNCTIONS (BIG DATA SAFE)
39) Rank cities by total revenue

In [ ]:

w_city = Window.orderBy(F.col("revenue").desc())
ranked_cities = rev_city.withColumn("rev_rank", F.dense_rank().over(w_city))
ranked_cities.show(truncate=False)


40) Rank products within each category by revenue

In [ ]:

rev_prod_cat = df_completed.groupBy("category", "product").agg(F.sum("amount_clean").alias("revenue"))
w_prod_in_cat = Window.partitionBy("category").orderBy(F.col("revenue").desc())
ranked_prod_in_cat = rev_prod_cat.withColumn("rank_in_category", F.dense_rank().over(w_prod_in_cat))
ranked_prod_in_cat.show(100, truncate=False)


41) Identify the top product per category

In [ ]:

top_prod_per_cat = ranked_prod_in_cat.filter(F.col("rank_in_category") == 1)
top_prod_per_cat.show(truncate=False)


42) Identify top 3 cities using window functions

In [ ]:

top3_cities = ranked_cities.filter(F.col("rev_rank") <= 3)
top3_cities.show(truncate=False)


PHASE 10 — CACHING & REUSE
43) Identify DataFrames reused multiple times

In [ ]:

df_completed.cache()
rev_city.cache(); rev_cat.cache(); rev_prod.cache()


45) Re-run analytics and observe performance

In [ ]:

start = time.time()
rev_city.count(); rev_cat.count(); rev_prod.count()


46)Unpersist when cache is no longer needed

In [ ]:
rev_city.unpersist(); rev_cat.unpersist(); rev_prod.unpersist(); df_completed.unpersist()

47) Explain why over-caching is dangerous

Over-caching consumes memory, can evict other useful blocks, increase GC, and degrade overall job throughput when many datasets are cached unnecessarily.

PHASE 11 — FILE FORMAT STRATEGY
48) Write the cleaned order-level dataset to Parquet

In [ ]:
base_path = "/content/"
out_parquet = base_path + "orders_cleaned_parquet"
df_completed.write.mode("overwrite").parquet(out_parquet)

49) Partition the Parquet output by city

In [ ]:

out_parquet_partitioned = base_path + "orders_cleaned_parquet_by_city"
df_completed.write.mode("overwrite").partitionBy("city").parquet(out_parquet_partitioned)



50) Write aggregated analytics to ORC

In [ ]:

out_orc_city = base_path + "city_revenue.orc"
out_orc_cat  = base_path + "category_revenue.orc"

rev_city.write.mode("overwrite").orc(out_orc_city)
rev_cat.write.mode("overwrite").orc(out_orc_cat)


51) Read both formats back and validate schema

In [ ]:

read_back_parquet = spark.read.parquet(out_parquet_partitioned)
read_back_orc_city = spark.read.orc(out_orc_city)

read_back_parquet.printSchema()
read_back_orc_city.printSchema()


PHASE 12 — DEBUGGING & FAILURE SCENARIOS
53) Explain why the following line breaks pipelines:

In [ ]:
df_filtered = df_completed.filter(F.col("amount_clean") > 50000)
df_filtered.show()

54) Create a scenario that produces a NoneType error

In [ ]:
df_tmp = df_completed.filter(F.col("amount_clean") > 50000)
df_tmp.show()
df_tmp.select("order_id").show()

PHASE 13 — FINAL VALIDATION
57) Validate no nulls in critical columns

In [ ]:

critical_cols = ["order_id","customer_id","city","category","product","amount_clean","order_date_clean","status"]
null_checks = {c: df_completed.filter(F.col(c).isNull()).count() for c in critical_cols}
null_checks


58) Confirm correct data types for all columns

In [ ]:
df_completed.printSchema()

59) Validate final record count

In [ ]:

final_count = df_completed.count()
print("Final record count (Completed, cleaned, deduped):", final_count)
